# Chạy so sánh năm mô hình

Structural/Raw Transformer, CANBERT+ adapted, CANLP-char-DNN adapted và CrySyS-AssocRules/W64. Mặc định CANBERT+ 768/6/12; đầu vào khác, supervision và thay đổi giao thức được ghi trong README. Chạy notebook 06 trước. **Pilot nhỏ chỉ xác minh phần mềm; full run mới là nghiên cứu.**

In [1]:
from pathlib import Path
import os, subprocess, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
PROJECT_HINT = Path('/content/drive/MyDrive/00_Research')  # sửa nếu đặt dự án nơi khác
candidates = [PROJECT_HINT, Path.cwd(), *Path.cwd().parents]
CODE = next((p / 'code' if (p / 'code/comparison/config.json').exists() else p
             for p in candidates if (p / 'code/comparison/config.json').exists() or
             (p / 'comparison/config.json').exists()), None)
assert CODE is not None, 'Đặt PROJECT_HINT tới thư mục chứa code/'
CODE = CODE.resolve()
CACHE = Path('/content/ctt_compare_cache') if IN_COLAB else CODE / 'results/comparison_data_v1'
OUTPUT = CODE / 'results/comparison_content_v1'
PILOT = CODE / 'results/comparison_pilot'
print('Code:', CODE, '\nCache:', CACHE, '\nResults:', OUTPUT)
def run_stage(stage, output=OUTPUT, extra=()):
    subprocess.run([sys.executable, str(CODE/'scripts/run_comparison.py'), '--stage', stage,
                    '--cache', str(CACHE), '--output', str(output), *map(str, extra)], check=True)


Code: F:\00_Research\code 
Cache: F:\00_Research\code\results\comparison_data_v1 
Results: F:\00_Research\code\results\comparison_content_v1


In [2]:
import torch, json
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
run_stage('plan')

Device: NVIDIA GeForce RTX 5060


## Pilot nguồn
Chỉ train/validation nguồn Set 1, seed đầu, mô hình nhỏ và một epoch. Không dùng A–D để chọn đối thủ. Chạy lại sẽ kiểm tra cell COMPLETE rồi bỏ qua; các cell FAILED/RUNNING tạo attempt mới.

In [4]:
run_stage('pilot', output=PILOT)

In [5]:
from pathlib import Path
checks = [json.loads(p.read_text()) for p in PILOT.glob('set_*/seed_*/*/attempt_*/manifest.json')]
complete = [m for m in checks if m['status']=='COMPLETE']
assert len({m['method'] for m in complete})==5, 'Cần đủ năm mô hình pilot thành công'
assert all(m['software_validation_only'] for m in complete)
print('Pilot PASS: có thể bắt đầu full run.')

Pilot PASS: có thể bắt đầu full run.


## Chạy đủ ma trận
4 set × 3 seed × 5 mô hình = 60 đơn vị IDS; CANBERT+ có thêm pretrain bên trong. Cần nơi lưu bền vững cho checkpoint và prediction/frame scores. Mỗi lần ngắt phải chạy lại notebook 06 để phục hồi cache local Colab, rồi gọi lại cùng lệnh dưới đây. Không đổi seed/model/config trong output hiện tại. Không có khôi phục từ epoch dở dang.

In [6]:
run_stage('run')

CalledProcessError: Command '['f:\\00_Research\\.venv\\Scripts\\python.exe', 'F:\\00_Research\\code\\scripts\\run_comparison.py', '--stage', 'run', '--cache', 'F:\\00_Research\\code\\results\\comparison_data_v1', '--output', 'F:\\00_Research\\code\\results\\comparison_content_v1']' returned non-zero exit status 4294967295.

Nếu gặp lỗi, giữ nguyên log và manifest FAILED. Không sửa COMPLETE hoặc xóa kết quả kém. Notebook 08 chỉ tổng hợp khi đủ tất cả cell. Đối chứng Relational Modelling là hạng mục tái lập tiếp theo, chưa nằm trong năm mô hình này.